# Waste Right — household organic-waste segmentation

Human-labelled TU Wien household bin images. Three pixel classes: background, organic and non-organic. This is a separate experimental model; it does not identify individual recyclable materials. Results are within one household, not a phone-camera performance claim.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
folder=Path("/content/drive/MyDrive/Waste Right Model Training/household-v2")


## Verified training package
The archive contains resized images, original-derived masks, grouped splits and publisher attribution. Check the package SHA-256 before extraction. Training independently audits image/mask hashes and split isolation.


In [ ]:
import hashlib,json,zipfile
manifest=json.loads((folder/"package-manifest.json").read_text())
archive=folder/manifest["name"]
assert hashlib.sha256(archive.read_bytes()).hexdigest()==manifest["sha256"]
work=Path("/content/household-v2")
work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (work/name).resolve().is_relative_to(work.resolve())
    z.extractall(work)
print((work/"prepared/report.json").read_text())


## Train and resume
25 epochs maximum, patience 6. The best validation foreground mean IoU selects the checkpoint. The test split is evaluated once after selection. Checkpoints and metrics are written to Drive. Re-running resumes compatible saved state.


In [ ]:
import subprocess,sys
output=folder/"runs/deeplab-v1"
subprocess.run([sys.executable,"-u",str(work/"train_household.py"),"--data",str(work/"prepared"),"--output",str(output),"--epochs","25","--batch-size","4","--resume","--require-cuda"],check=True)


## Held-out results
Report organic and contaminant IoU separately. IoU and the first detector’s mAP measure different tasks. Do not compare them as an accuracy improvement.


In [ ]:
print((output/"evaluation.json").read_text())
print((output/"model_card.json").read_text())


## Source and limits
Florian Wolling and Gabor Pal, TU Wien, version 1.0.0, CC BY 4.0: https://doi.org/10.48436/27k90-dvw73

450 images from one family’s fixed bin camera. Groups combine capture days and exact/dHash-near duplicates. More households and phone camera evaluation are still needed. Pixel area is not mass, compostability, safety or a disposal recommendation. The checkpoint remains outside the app pending review and calibration.
